In [0]:
 from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/s3_pipline/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "arc")
catalog = dbutils.widgets.get("catalog")
print(catalog)

arc


In [0]:
dbutils.widgets.text("catalog", "arc")
dbutils.widgets.text("data_source", "customers")
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sport-bar-try/{data_source}/*.csv"

print(catalog)
print(data_source)
print(base_path)

arc
customers
s3://sport-bar-try/customers/*.csv


In [0]:
try:
    files = dbutils.fs.ls(base_path)

    csv_files = [f for f in files if f.path.endswith(".csv")]

    if not csv_files:
        raise Exception(f"No CSV files found in source path: {base_path}")

except Exception as e:
    raise Exception(f"Source validation failed: {base_path} | {str(e)}")


df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

display(df.limit(10))

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-5962861524190058>, line 2
      1 try:
----> 2     files = dbutils.fs.ls(base_path)
      4     csv_files = [f for f in files if f.path.endswith(".csv")]

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:57, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     55 error_exception = ExecutionError(str(e))
---> 57 raise patch_exception_with_error_details(
     58     error_exception,
     59     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore[attr-defined]
     60 ) from None

ExecutionError: (com.databricks.sql.io.CloudFileNotFoundException) No such file or directory: s3://sport-bar-try/customers/*.csv

JVM stacktrace:
com.databricks.sql.io.CloudFileNotFoundException
	at com.databricks.sql.io.NativeIOUtils.translateToSQLStateExceptions(NativeIO

In [0]:
(
    df.write 
    .format("delta") 
    .option("delta.enableChangeDataFeed", "true") 
    .mode("overwrite") 
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
)

### Silver Processing

In [0]:
df_bronze = spark.sql(
    f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};"
)

df_bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-09-09 11:53:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-09-09 11:53:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-09-09 11:53:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-09-09 11:53:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-09-09 11:53:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengalore|2026-09-09 11:53:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-09-09 11:53:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-09-09 11:53:...|customers.csv|     1404|
|     7891

In [0]:
df_bronze.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter("count > 1")
df_duplicates.show()
df_duplicates.count()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|     789321|    2|
|     789503|    2|
|     789522|    2|
|     789603|    2|
+-----------+-----+



4

In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

Rows before duplicates dropped:  39
Rows after duplicates dropped:  35


In [0]:
# check those values
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-09-09T11:53:54.769Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-09-09T11:53:54.769Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-09-09T11:53:54.769Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-09-09T11:53:54.769Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-09-09T11:53:54.769Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-09-09T11:53:54.769Z,customers.csv,1404


In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

df_silver.display()

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-09-09T11:53:54.769Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-09-09T11:53:54.769Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-09-09T11:53:54.769Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-09-09T11:53:54.769Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-09-09T11:53:54.769Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-09-09T11:53:54.769Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-09-09T11:53:54.769Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-09-09T11:53:54.769Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-09-09T11:53:54.769Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-09-09T11:53:54.769Z,customers.csv,1404


In [0]:
df_silver.select('city').distinct().show()

+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|      NULL|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+



In [0]:
# # typo dictionary
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

# typos → correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
df_silver.show()

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-09-09 11:53:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-09-09 11:53:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-09-09 11:53:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-09-09 11:53:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-09-09 11:53:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengaluru|2026-09-09 11:53:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-09-09 11:53:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-09-09 11:53:...|customers.csv|     1404|
|     7891

In [0]:
df_silver.select('city').distinct().show()

+---------+
|     city|
+---------+
|Bengaluru|
|Hyderabad|
|New Delhi|
|     NULL|
+---------+



In [0]:
df_silver.select('customer_name').distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      FitFuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|HydroBoost Nutrition|
|MacroBite Superfoods|
|MacroBite superfoods|
|      PowerSnack Hub|
|      PowerSnack hub|
|   SprintX nutrition|
|   SprintX Nutrition|
|    ZenAthlete foods|
|    ZenAthlete Foods|
|Peak performance ...|
|Peak Performance ...|
| PrimeFuel Nutrition|
|       Recovery Lane|
|      StaminaX Store|
|EliteAthlete Nutr...|
|      GamePlan Foods|
|   Champion's choice|
+--------------------+
only showing top 20 rows


In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.initcap(F.col("customer_name"))
)
# df_silver.show()
df_silver.select('customer_name').distinct().display()
df_silver

customer_name
Fitfuel Market
Athlete's Choice Store
Endurance Foods
Hydroboost Nutrition
Macrobite Superfoods
Powersnack Hub
Sprintx Nutrition
Zenathlete Foods
Peak Performance Store
Primefuel Nutrition


DataFrame[customer_id: int, customer_name: string, city: string, read_timestamp: timestamp, file_name: string, file_size: bigint]

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

+-----------+-------------------+----+--------------------------+-------------+---------+
|customer_id|customer_name      |city|read_timestamp            |file_name    |file_size|
+-----------+-------------------+----+--------------------------+-------------+---------+
|789403     |Sprintx Nutrition  |NULL|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789603     |Recovery Lane      |NULL|2026-09-09 11:53:54.769181|customers.csv|1404     |
+-----------+-------------------+----+--------------------------+-------------+---------+



In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+--------------------------+-------------+---------+
|customer_id|customer_name      |city     |read_timestamp            |file_name    |file_size|
+-----------+-------------------+---------+--------------------------+-------------+---------+
|789401     |Sprintx Nutrition  |Bengaluru|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789402     |Sprintx Nutrition  |Hyderabad|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789403     |Sprintx Nutrition  |NULL     |2026-09-09 11:53:54.769181|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL     |2026-09-09 11:53:54.769181|customers.csv|1404     |
|789421     |Zenathlete Foods   |Hyderabad|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789422     |Zenathlete Foods   |New Delhi|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789520     |Primefuel Nutrition|Bengaluru|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL     |2026-09

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

customer_id,fixed_city
789403,New Delhi
789420,Bengaluru
789521,Hyderabad
789603,Hyderabad


In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

In [0]:
# Sanity Checks

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+--------------------------+-------------+---------+
|customer_id|customer_name      |city     |read_timestamp            |file_name    |file_size|
+-----------+-------------------+---------+--------------------------+-------------+---------+
|789601     |Recovery Lane      |Bengaluru|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789420     |Zenathlete Foods   |Bengaluru|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789421     |Zenathlete Foods   |Hyderabad|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789520     |Primefuel Nutrition|Bengaluru|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789403     |Sprintx Nutrition  |New Delhi|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789521     |Primefuel Nutrition|Hyderabad|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789401     |Sprintx Nutrition  |Bengaluru|2026-09-09 11:53:54.769181|customers.csv|1404     |
|789402     |Sprintx Nutrition  |Hyderabad|2026-09

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

None


In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(5))

customer_id,customer_name,city,read_timestamp,file_name,file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-09-09T11:53:54.769Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-09-09T11:53:54.769Z,customers.csv,1404,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-09-09T11:53:54.769Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-09-09T11:53:54.769Z,customers.csv,1404,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-09-09T11:53:54.769Z,customers.csv,1404,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold Processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "ARC.gold.dim_customers")
df_child_customers = spark.table("ARC.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
# get the latest version of the table
df = spark.table(f"{catalog}.{gold_schema}.sb_dim_{data_source}")
display(df)

customer_id,customer_name,city,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods,Bengaluru,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market,Hyderabad,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods,Hyderabad,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition
